In [0]:
%pip install pyrosettacolabsetup pyrosetta-installer altair

In [0]:
import pyrosetta_installer
pyrosetta_installer.install_pyrosetta(use_setup_py_package= True)

In [0]:
%restart_python

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

### PyRosetta Metrics & Holo vs Apo Interface RMSD

In [0]:
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics") # add path to folder containing scoring script
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/Scfv_Toolkit")
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/MotifScaffold/utils")

In [0]:
dbutils.widgets.text("path_design_campaign", "")
path_design_campaign = dbutils.widgets.get("path_design_campaign")
print("Path Design Campaign: ", path_design_campaign)

In [0]:
dbutils.widgets.text("path_task_yaml", "")
path_task_yaml = dbutils.widgets.get("path_task_yaml")
print("Path Task Yaml: ", path_task_yaml)

In [0]:
import pandas as pd
import os

path_boltz_metrics = os.path.join(path_design_campaign, "af2_passed_boltz_input")
list_df_apo = []
for filename in os.listdir(path_boltz_metrics):
    if (("apo_scored" in filename) and (filename.endswith(".csv"))):
        df_boltz_apo_single_full = pd.read_csv(os.path.join(path_boltz_metrics, filename))
        list_df_apo.append(df_boltz_apo_single_full)
    elif (("af2_boltz_holo_passed_median" in filename) and (filename.endswith(".csv"))) :
        df_boltz_holo = pd.read_csv(os.path.join(path_boltz_metrics, filename), index_col=0)

df_boltz_apo_full = pd.concat(list_df_apo).reset_index(drop = True)
df_boltz_apo = df_boltz_apo_full[df_boltz_apo_full['model_id'] == 0].reset_index(drop = True)
print("Number of designs in boltz holo: ", len(df_boltz_holo))
print("Number of designs in boltz apo: ", len(df_boltz_apo))
df_boltz_apo.head()

In [0]:
df_boltz_holo_apo = pd.merge(df_boltz_holo, df_boltz_apo, on = 'design_name', how = 'inner')
df_boltz_holo_apo

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize = (10,10))
sns.scatterplot(data = df_boltz_holo_apo, x = 'rmsd_af2_boltz_apo_binder', y = 'rmsd_af2_boltz_holo_binder', ax = ax)
ax.set_xlabel("AF2 Holo vs Boltz Apo (Aligned on Binder Chain & Measured Binder RMSD)")
ax.set_ylabel("AF2 Holo vs Boltz Holo (Aligned on Target Chain & Measured Binder RMSD)")
plt.show()

In [0]:
from utils import *
from RFDUtils import *

# 1. Extract Binder Contig
example_contig = df_boltz_holo_apo.iloc[0]['contig']
print("Example Contigs: ", example_contig)
binder_contig = extract_binder_contig(example_contig)
print("Binder_contigs: ", binder_contig)

# 2. Create Align & Measured Masks for aligning: RFD Holo Structure, Boltz Holo Structure, and Boltz Apo Structure
example_coords, atom_mask = extract_design_motif_coords(df_boltz_holo_apo.iloc[0]['path_boltz_apo_structure'], binder_contig, binder_chain_id= 'A', return_coords_mask = True)

In [0]:
import os # Ensure os is imported for _cpu_allocation

def _worker_wrapper(unique_args, **constant_args):
    """ Unpacks unique arguments for each parallel worker """
    pdb_file_path, unique_rmsd_input = unique_args
    
    return run_relaxation_and_physics_scoring_single_pdb(
        pdb_file_path=pdb_file_path,
        rmsd_inputs=unique_rmsd_input,
        **constant_args # Unpacks epi_residues, binder_chain_id, etc.
    )

def cpu_allocation() -> int:
    """Helper function to determine the optimal number of CPU workers."""
    try:
        # Prefer Linux CPU affinity (matches cluster allocations)
        return max(len(os.sched_getaffinity(0)), 1)
    except Exception:
        pass
    try:
        import multiprocessing
        return max(int(multiprocessing.cpu_count()), 1)
    except Exception:
        return max(int(os.cpu_count() or 1), 1)
cpu_allocation()

Extracting Align and Measure Masks for RMSD calculations

In [0]:
import yaml
with open(path_task_yaml, 'r') as file:
    task_config = yaml.safe_load(file)
    print(task_config)
# 1. Extract Inputs from Config
desired_motif_chain_id = task_config['motif_chain_id']

In [0]:
list_rmsd_inputs = []
for index, row in df_boltz_holo_apo.iterrows():

    # 0: Extract inputs from row
    contig = row['contig']
    filepath_apo = row['path_boltz_apo_structure']

    # 1. Extract Binder Contig
    print("Contig: ", contig)
    binder_contig = extract_binder_contig(contig)
    print("Binder_contigs: ", binder_contig)

    # 2. Create Align & Measured Masks for aligning: RFD Holo Structure, Boltz Holo Structure, and Boltz Apo Structure
    example_coords, align_mask = extract_design_motif_coords(filepath_apo, binder_contig, binder_chain_id= 'A', 
                                                             return_coords_mask = True, desired_motif_chain_id= "")
    example_coords, measure_mask = extract_design_motif_coords(filepath_apo, binder_contig, binder_chain_id= 'A', 
                                                               return_coords_mask = True, desired_motif_chain_id= "")
    
    list_rmsd_inputs.append({'filepath_apo': filepath_apo, 'align_mask': align_mask, 'measure_mask' : measure_mask})

In [0]:
import os
from multiprocessing import Pool
from functools import partial
from tqdm import tqdm
import pandas as pd
import EvalPhysics
from EvalPhysics import run_relaxation_and_physics_scoring_single_pdb

# Extract constant arguments/inputs to run relaxation and physics scoring
epi_contact_residues = task_config['hotspots'].split(",")
print("Epitope Contact Residues or Target Hotspots: ", epi_contact_residues)
cutoff = 4.5

# Extract unique arguments/inputs to run relaxation and physics scoring


binder_target_file_paths = list(df_boltz_holo_apo['path_boltz_holo_structure'])

# Bundle unique arguments for each parallel worker

tasks_inputs = list(zip(binder_target_file_paths, list_rmsd_inputs))

worker_func = partial(_worker_wrapper,
                      epi_residues = epi_contact_residues,
                      binder_chain_id = "A",
                      target_chain_id = "B",
                      linker = False)

workers = cpu_allocation()

all_interface_metrics = []

if workers <= 1 or len(tasks_inputs) <= 1:
        print("Running sequentially (1 worker)...")
        for task in tqdm(tasks_inputs, desc="Processing PDBs"):
            all_interface_metrics.append(worker_func(task))
else:
    print(f"Running in parallel with {workers} workers...")
    with Pool(workers) as pool:
        all_interface_metrics = list(tqdm(pool.imap(worker_func, tasks_inputs), total=len(tasks_inputs), desc="Processing PDBs"))

# Convert metrics into a Pandas DataFrame and add columns for ptm_apo, filepath_apo, and sequence

df_interface_metrics_boltz = pd.DataFrame(all_interface_metrics)
df_interface_metrics_boltz.head()


In [0]:
df_interface_metrics_boltz.sort_values(by = 'interface_holo_apo_rmsd', ascending = True)

In [0]:
df_interface_metrics_boltz.sort_values(by = 'interface_holo_apo_rmsd', ascending = True).head(30)

In [0]:
# Save PyRosetta Physics Based Calculations to CSV File
path_metrics_destination = os.path.join(path_design_campaign, "final_metrics")
df_interface_metrics_boltz.to_csv(os.path.join(path_metrics_destination, "physics_rmsd_metrics.csv"))

# Save Boltz Apo Full Metrics (Metrics from all 5 Diffusion Samples) to CSV File
df_boltz_apo_full.to_csv(os.path.join(path_metrics_destination, "boltz_apo_scored_designs.csv"))